In [1]:
print("Hello, ML World!")

Hello, ML World!


In [2]:
import pandas as pd

In [3]:
# Load input data
df = pd.read_csv("email.csv")

In [5]:
print(df.shape)

(5573, 2)


In [7]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [8]:
# Fill missing data (cleaning)
df["Category"] = df["Category"].fillna("").astype(str).str.strip().str.lower()
df["Message"]  = df["Message"].fillna("").astype(str)

# keep only valid labels
df = df[df["Category"].isin(["ham", "spam"])].copy()
print(df["Category"].value_counts())

Category
ham     4825
spam     747
Name: count, dtype: int64


In [9]:
# Store Y (labels) as numbers
df["y"] = df["Category"].map({"ham": 0, "spam": 1}).astype(int)

X_text = df["Message"]   # raw text input (X)
y      = df["y"]         # numeric label (Y)

In [10]:
X_text

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                 Will ü b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: Message, Length: 5572, dtype: object

In [11]:
y

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: y, Length: 5572, dtype: int64

In [12]:
# Train/Test split (store X_train, X_test, y_train, y_test)
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(len(X_train_text), len(X_test_text))

4457 1115


In [16]:
y_test

2825    0
3695    0
3904    0
576     1
2899    0
       ..
854     0
5044    0
2015    0
3380    0
785     0
Name: y, Length: 1115, dtype: int64

In [17]:
# Convert text to numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)

X_train = vectorizer.fit_transform(X_train_text)  # learns vocabulary + TF-IDF
X_test  = vectorizer.transform(X_test_text)

print(X_train.shape, X_test.shape)

(4457, 7398) (1115, 7398)


In [20]:
# “Add weights” (initial weights) + Run model
import numpy as np
from sklearn.linear_model import SGDClassifier

clf = SGDClassifier(loss="log_loss", penalty="l2", alpha=1e-4, random_state=42)

# This is where weights are initialized internally (near-zero/random).
# We don't manually set them; the library initializes them.
classes = np.array([0, 1])

In [23]:
# Find loss, compare loss, update weights (optimizer loop)
from sklearn.metrics import log_loss, accuracy_score

for epoch in range(1, 10):
    # Update weights using the optimizer (SGD)
    if epoch == 1:
        clf.partial_fit(X_train, y_train, classes=classes)
    else:
        clf.partial_fit(X_train, y_train)

    # Output probabilities (model output)
    p_train = clf.predict_proba(X_train)
    p_test  = clf.predict_proba(X_test)

    # Loss (how wrong)
    train_loss = log_loss(y_train, p_train)
    test_loss  = log_loss(y_test, p_test)

    # Compare loss + track accuracy
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc  = accuracy_score(y_test, clf.predict(X_test))

    print(
        f"Epoch {epoch:>2}: "
        f"train loss={train_loss:.4f}, test loss={test_loss:.4f}, "
        f"train acc={train_acc:.3f}, test acc={test_acc:.3f}"
    )


Epoch  1: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  2: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  3: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  4: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  5: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  6: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  7: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  8: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978
Epoch  9: train loss=0.0774, test loss=0.1015, train acc=0.983, test acc=0.978


In [24]:
# Inspect learned weights (optional but very useful)
import numpy as np

feature_names = vectorizer.get_feature_names_out()
weights = clf.coef_[0]   # learned weights

top_spam_idx = np.argsort(weights)[-15:][::-1]
top_ham_idx  = np.argsort(weights)[:15]

print("Top SPAM words:")
for i in top_spam_idx[:10]:
    print(feature_names[i], weights[i])

print("\nTop HAM words:")
for i in top_ham_idx[:10]:
    print(feature_names[i], weights[i])


Top SPAM words:
txt 5.787799980247978
uk 4.921795661858951
claim 4.563306403292583
mobile 4.4537112301381825
www 4.377368751806656
service 4.248650950946746
stop 4.2031641870169745
reply 4.103389100784303
150p 3.9838201171990453
text 3.445688249584843

Top HAM words:
ok -2.3310279756275403
gt -2.2918490165797323
lt -2.2849510296983486
ll -2.261277598946058
da -1.9098785907766127
home -1.797611420231661
got -1.7454161274933104
come -1.7449681978273375
lor -1.5987186490941447
sorry -1.5798005362178298
